# DSCI 552 Homework 3

Name: Zhisong Han  
GitHub Username: zshan666  
USC ID: 5284161601

In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### 1(a) Load the AReM Data

In [2]:
data_path = "../data/AReM"
activities = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]
for activity in activities:
    folder_path = os.path.join(data_path, activity)
    files = sorted([
        f for f in os.listdir(folder_path)
        if f.endswith(".csv")
    ])
    print(activity, len(files))

bending1 7
bending2 6
cycling 15
lying 15
sitting 15
standing 15
walking 15


### 1(b) Train/Test Split

In [3]:
train_files = []
test_files = []
for activity in activities:
    folder_path = os.path.join(data_path, activity)
    files = sorted(
        [f for f in os.listdir(folder_path) if f.endswith(".csv")],
        key=lambda x: int(x.replace("dataset", "").replace(".csv", ""))
    )
    if activity in ["bending1", "bending2"]:
        test_names = ["dataset1.csv", "dataset2.csv"]
    else:
        test_names = ["dataset1.csv", "dataset2.csv", "dataset3.csv"]
    for file in files:
        full_path = os.path.join(folder_path, file)

        if file in test_names:
            test_files.append((activity, file, full_path))
        else:
            train_files.append((activity, file, full_path))
print("Training instances:", len(train_files))
print("Testing instances:", len(test_files))

Training instances: 69
Testing instances: 19


### 1(c) Feature Extraction

#### 1(c)(i) Common Time-Domain Features

Some commonly used time-domain features for time series classification are minimum, maximum, mean, median, standard deviation, variance, first quartile, third quartile, range, and interquartile range.

#### 1(c)(ii) Extract Time-Domain Features

In [17]:
column_names = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]
def read_dataset(file_path):
    rows = []
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.replace(",", " ").split()
            if len(parts) >= 7:
                rows.append(parts[:7])
    data = pd.DataFrame(
        rows,
        columns=column_names
    ).apply(pd.to_numeric)
    expected_times = np.arange(0, 120000, 250)
    if len(data) < 480:
        data = (
            data.set_index("time")
            .reindex(expected_times)
            .interpolate(method="linear")
            .reset_index()
            .rename(columns={"index": "time"})
        )
    return data

In [19]:
def extract_features(data):
    features = {}
    time_series = column_names[1:]
    for i, col in enumerate(time_series, start=1):
        x = data[col]
        features[f"min{i}"] = x.min()
        features[f"max{i}"] = x.max()
        features[f"mean{i}"] = x.mean()
        features[f"median{i}"] = x.median()
        features[f"std{i}"] = x.std()
        features[f"q1_{i}"] = x.quantile(0.25)
        features[f"q3_{i}"] = x.quantile(0.75)
    return features

In [20]:
feature_rows = []
train_keys = {
    (activity, file)
    for activity, file, _ in train_files
}
for activity, file, file_path in all_files:
    data = read_dataset(file_path)
    row = {
        "activity": activity,
        "dataset": file,
        "set": "train" if (activity, file) in train_keys else "test"
    }
    row.update(extract_features(data))
    feature_rows.append(row)
features_df = pd.DataFrame(feature_rows)
print("Shape:", features_df.shape)
features_df.head()

Shape: (88, 45)


,activity,dataset,set,min1,max1,mean1,median1,std1,q1_1,q3_1,...,std5,q1_5,q3_5,min6,max6,mean6,median6,std6,q1_6,q3_6
0,bending1,dataset3.csv,train,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.00,0.94
1,bending1,dataset4.csv,train,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.00,1.00
2,bending1,dataset5.csv,train,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.00,0.50
3,bending1,dataset6.csv,train,37.00,48.00,43.454958,43.25,1.386098,42.50,45.00,...,2.488862,22.2500,24.00,0.0,5.26,0.679646,0.50,0.622534,0.43,0.87
4,bending1,dataset7.csv,train,36.25,48.00,43.969125,44.50,1.618364,43.31,44.67,...,3.318301,20.5000,23.75,0.0,2.96,0.555312,0.49,0.487826,0.00,0.83


In [21]:
print("Number of instances:", len(features_df))
print("Number of extracted features:", features_df.shape[1] - 3)
print("\nTrain/Test counts:")
print(features_df["set"].value_counts())

Number of instances: 88
Number of extracted features: 42

Train/Test counts:
set
train    69
test     19
Name: count, dtype: int64


### 1(c)(iii) Standard Deviations and 90% Bootstrap Confidence Intervals

I calculate the standard deviation of each extracted feature and use bootstrap resampling to estimate a 90% confidence interval.

In [22]:
feature_columns = features_df.columns[3:]
rng = np.random.default_rng(42)
bootstrap_results = []
for feature in feature_columns:
    values = features_df[feature].to_numpy()
    std_estimate = np.std(values, ddof=1)
    bootstrap_stds = []
    for _ in range(2000):
        sample = rng.choice(
            values,
            size=len(values),
            replace=True
        )
        bootstrap_stds.append(
            np.std(sample, ddof=1)
        )
    lower = np.percentile(bootstrap_stds, 5)
    upper = np.percentile(bootstrap_stds, 95)
    bootstrap_results.append({
        "Feature": feature,
        "Std": std_estimate,
        "90% CI Lower": lower,
        "90% CI Upper": upper
    })
bootstrap_df = pd.DataFrame(bootstrap_results)
bootstrap_df

,Feature,Std,90% CI Lower,90% CI Upper
0,min1,9.569975,8.252809,10.769540
1,max1,4.394362,3.346962,5.293193
2,mean1,5.335703,4.677227,5.883952
3,median1,5.440054,4.819548,6.003508
4,std1,1.772155,1.570051,1.959516
5,q1_1,6.153590,5.568901,6.657172
6,q3_1,5.138925,4.313308,5.862059
7,min2,0.000000,0.000000,0.000000
8,max2,5.062729,4.652329,5.414498
9,mean2,1.574170,1.400251,1.703019


In [23]:
print("Number of features:", len(bootstrap_df))

Number of features: 42


### 1(c)(iv) Most Important Time-Domain Features

I would choose minimum, mean, and maximum as the three most important time-domain features. The minimum and maximum describe the range of the activity, while the mean gives the general level of the signal. Together, they give a simple summary of both the center and the extremes of each time series.

## 2. ISLR 3.7.4

### (a)

The cubic regression should have a lower or equal training RSS because it is more flexible and includes the linear model as a special case.

### (b)

The linear regression should probably have a lower test RSS because the true relationship is linear. The cubic model may fit some extra noise and overfit.

### (c)

The cubic regression should still have a lower or equal training RSS because it is more flexible than the linear model.

### (d)

There is not enough information to tell. If the true relationship is very nonlinear, the cubic model may have a lower test RSS. If it is only slightly nonlinear, the linear model could still perform better.